In [ ]:
library(tidyverse)
library(pheatmap)
library(data.table)

reverse_complement <- function(dna_seq) {
  complement <- c("A" = "T", "T" = "A", "C" = "G", "G" = "C")
  nucleotides <- unlist(strsplit(dna_seq, ""))
  complement_nucleotides <- complement[nucleotides]
  reverse_complement_seq <- paste(rev(complement_nucleotides), collapse = "")
  return(reverse_complement_seq)
}

twist_barcodes <- read_csv("/mnt/dawnccle2/melange/data/guide_library/20230130_twist_library_v3.csv") %>%
  mutate(barcodeRevcomp = sapply(barcode, reverse_complement))

######## Look at rmats stats ########
combined_psi <- read_tsv("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/MUT2_PSI_combined_output_indiv.tsv")
calculate_ratio <- function(I, S) {
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  ratio <- I_values / (I_values + S_values)
  return(paste(round(ratio,3), collapse = ","))
}

calculate_average <- function(PSI){
  PSI_values <- as.numeric(unlist(strsplit(PSI, ",")))
  average <- mean(PSI_values)
  return(round(average, 3))
}

calculate_average_count_sum <- function(I, S){
  I_values <- as.numeric(unlist(strsplit(I, ",")))
  S_values <- as.numeric(unlist(strsplit(S, ",")))
  total_sum <- I_values + S_values
  average_count_sum <- mean(total_sum)
  return(round(average_count_sum, 0))
}

# Apply the function to the data frame
combined_psi <- combined_psi %>%
  mutate(
    PSI1 = mapply(calculate_ratio, I1, S1),
    PSI2 = mapply(calculate_ratio, I2, S2)
  ) %>% 
  mutate(
    PSI1_average = mapply(calculate_average, PSI1),
    PSI2_average = mapply(calculate_average, PSI2)
  ) %>%
  mutate(PSI_diff = PSI1_average - PSI2_average) %>% 
  mutate(
    count_sum_average1 = mapply(calculate_average_count_sum, I1, S1),
    count_sum_average2 = mapply(calculate_average_count_sum, I2, S2)
  ) %>% mutate(PSI_ratio = PSI1_average / PSI2_average) %>% 
  mutate(PSI_reverse_ratio = (1-PSI1_average)/(1-PSI2_average))

wanted_pairs <- c(
#'MUT2_CH3-1_A1_CH3-1_A2',
'MUT2_CH3-1_A1_FUBP1_B12',
'MUT2_CH3-1_A1_FUBP1_C5',
'MUT2_CH3-1_A1_RBM10_C8',
'MUT2_CH3-1_A1_RBM10_G4',
'MUT2_CH3-1_A1_RBM5_A2',
'MUT2_CH3-1_A1_RBM5_A3',
'MUT2_CH3-1_A1_ZRSR2_F8',
'MUT2_CH3-1_A1_ZRSR2_G9',
'MUT2_CH3-1_A2_FUBP1_B12',
'MUT2_CH3-1_A2_FUBP1_C5',
'MUT2_CH3-1_A2_RBM10_C8',
'MUT2_CH3-1_A2_RBM10_G4',
'MUT2_CH3-1_A2_RBM5_A2',
'MUT2_CH3-1_A2_RBM5_A3',
'MUT2_CH3-1_A2_ZRSR2_F8',
'MUT2_CH3-1_A2_ZRSR2_G9')

# Get sequences in folders MUT2_CH3-1_A1
all_ch3_a1 <- combined_psi_filtered %>% filter(grepl("MUT2_CH3-1_A1", Folder)) 
chr3_a1_seq_count <- all_ch3_a1 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a1 <- chr3_a1_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also exclude sequences in MUT2_CH3-1_A2
all_ch3_a2 <- combined_psi_filtered %>% filter(grepl("MUT2_CH3-1_A2", Folder)) 
chr3_a2_seq_count <- all_ch3_a2 %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
arrange(desc(count))

seq_to_exclude_a2 <- chr3_a2_seq_count %>% 
filter(count >=4) %>% 
pull(ExonID)

# Also look at diff between A1 and A2
control_de <- combined_psi %>% 
filter(Folder == "MUT2_CH3-1_A1_CH3-1_A2") %>% 
filter(count_sum_average1 >30) %>% 
filter(count_sum_average2 > 30) %>% 
mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

seq_to_exclude_control <- control_de %>% 
pull(ExonID) %>% 
unique()

all_seq_to_exclude <- c(seq_to_exclude_a1, seq_to_exclude_a2, seq_to_exclude_control)

combined_psi_filtered <- combined_psi %>% 
  filter(!ExonID %in% all_seq_to_exclude) %>% 
  filter(Folder %in% wanted_pairs) %>%
  filter(count_sum_average1 >30) %>% 
  filter(count_sum_average2 > 30) %>% 
  mutate(log2_PSI_ratio = log2(PSI_ratio), log2_PSI_reverse_ratio = log2(PSI_reverse_ratio)) %>% 
  separate(ExonID, sep = "__", into =c("index", "offset"), remove = FALSE) %>%
  separate(offset, into = c("skipped_exon_start", "skipped_exon_end", "downstream_exon_start"), sep = ":", remove = FALSE) %>%
  filter(abs(as.integer(skipped_exon_start)) != 1 & abs(as.integer(skipped_exon_end)) != 1) 

######## Process and Save the Individual Sequences as Fasta ######
# Define a function to process and save sequences
process_and_save <- function(df, name) {
  # Define file paths
  upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_upstreamIntronSeq_adj.fasta")
  skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_skippedExonSeq_adj.fasta")
  downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", name, "_downstreamIntronSeq_adj.fasta")
  
  # Write upstream intron sequences
  fileConn <- file(upstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write skipped exon sequences
  fileConn <- file(skipped_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Write downstream intron sequences
  fileConn <- file(downstream_fasta, "w")
  apply(df, 1, function(row) {
    cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq_adj"], "\n"), file = fileConn)
  })
  close(fileConn)
  
  # Save CSV file
  csv_path <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_", name, "_seq.csv")
  write_csv(df, csv_path)
  
  # Message to indicate completion
  cat("FASTA and CSV files saved for:", name, "\n")
}

# Process and save each dataset
unique_folders <- unique(combined_psi_filtered$Folder)

# Create a list of datasets for all unique folders
datasets <- list()
for (folder in unique_folders) {
   # folder_name <- gsub("MUT2_", "", folder)  # Remove the MUT2_ prefix if present
  datasets[[folder]] <- combined_psi_filtered %>% filter(Folder == folder)
}

# Print the number of datasets to be processed
cat("Processing", length(datasets), "unique datasets\n")

for (name in names(datasets)) {
  df <- datasets[[name]] %>%
    left_join(twist_barcodes, by = c("index" = "ID")) %>%
    mutate(
      upstream_offset = as.integer(skipped_exon_start),
      downstream_offset = as.integer(skipped_exon_end),
      const_offset = as.integer(downstream_exon_start),
      upstreamIntron_len = nchar(upstreamIntronSeq),
      downstreamIntron_len = nchar(downstreamIntronSeq),
      skippedExon_len = nchar(skippedExonSeq),
      upstreamIntron_len_adj = upstreamIntron_len + upstream_offset,
      downstreamIntron_len_adj = downstreamIntron_len - downstream_offset,
      skippedExon_len_adj = skippedExon_len - upstream_offset + downstream_offset,
      upstreamIntronSeq_adj = substr(librarySequence, 1, upstreamIntron_len_adj),
      skippedExonSeq_adj = substr(librarySequence, upstreamIntron_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj),
      downstreamIntronSeq_adj = substr(librarySequence, upstreamIntron_len_adj + skippedExon_len_adj + 1, upstreamIntron_len_adj + skippedExon_len_adj + downstreamIntron_len_adj)
    )
  
  process_and_save(df, name)
}

cat("Processing complete for all datasets.\n")

# Make the "background sequence" file 
upstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_upstreamIntronSeq_adj.fasta")
skipped_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_skippedExonSeq_adj.fasta")
downstream_fasta <- paste0("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/", "background_downstreamIntronSeq_adj.fasta")

# Write upstream intron sequences
fileConn <- file(upstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["upstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write skipped exon sequences
fileConn <- file(skipped_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["skippedExonSeq"], "\n"), file = fileConn)
})
close(fileConn)

# Write downstream intron sequences
fileConn <- file(downstream_fasta, "w")
apply(twist_barcodes, 1, function(row) {
  cat(paste0(">", paste0(row["barcode"], "_", row["offset"]), "\n", row["downstreamIntronSeq"], "\n"), file = fileConn)
})
close(fileConn)

Rows: 46372 Columns: 7
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (7): ID, barcode, upstreamIntronSeq, skippedExonSeq, downstreamIntronSeq...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
Rows: 65535 Columns: 10
── Column specification ────────────────────────────────────────────────────────
Delimiter: "\t"
chr (6): ExonID, I1, S1, I2, S2, Folder
dbl (4): I_len, S_len, PValue, FDR

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


Processing 16 unique datasets
FASTA and CSV files saved for: MUT2_CH3-1_A1_FUBP1_B12 
FASTA and CSV files saved for: MUT2_CH3-1_A1_FUBP1_C5 
FASTA and CSV files saved for: MUT2_CH3-1_A1_RBM10_C8 
FASTA and CSV files saved for: MUT2_CH3-1_A1_RBM10_G4 
FASTA and CSV files saved for: MUT2_CH3-1_A1_RBM5_A2 
FASTA and CSV files saved for: MUT2_CH3-1_A1_RBM5_A3 
FASTA and CSV files saved for: MUT2_CH3-1_A1_ZRSR2_F8 
FASTA and CSV files saved for: MUT2_CH3-1_A1_ZRSR2_G9 
FASTA and CSV files saved for: MUT2_CH3-1_A2_FUBP1_B12 
FASTA and CSV files saved for: MUT2_CH3-1_A2_FUBP1_C5 
FASTA and CSV files saved for: MUT2_CH3-1_A2_RBM10_C8 
FASTA and CSV files saved for: MUT2_CH3-1_A2_RBM10_G4 
FASTA and CSV files saved for: MUT2_CH3-1_A2_RBM5_A2 
FASTA and CSV files saved for: MUT2_CH3-1_A2_RBM5_A3 
FASTA and CSV files saved for: MUT2_CH3-1_A2_ZRSR2_F8 
FASTA and CSV files saved for: MUT2_CH3-1_A2_ZRSR2_G9 
Processing complete for all datasets.


NULL

NULL

NULL

In [52]:
# Plot the volcano plot
# save the plot
p1 <- ggplot(combined_psi_filtered, aes(log2(PSI_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 2) +
  theme_classic() +
  ggtitle("PSI ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_volcano_plot_PSI_ratio.png", p1, width = 20, height = 6)

# Plot PSI_reverse_ratio
p2 <- ggplot(combined_psi_filtered, aes(log2(PSI_reverse_ratio), -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 2) +
  theme_classic() +
  ggtitle("PSI reverse ratio")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_volcano_plot_PSI_reverse_ratio.png", p2, width = 20, height = 6)

# Also plot the PSI_dff
p3 <- ggplot(combined_psi_filtered, aes(PSI_diff, -log10(FDR))) + 
  geom_point() + 
  facet_wrap(~Folder, nrow = 2) +
  theme_classic() +
  ggtitle("PSI diff")
ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_volcano_plot_PSI_diff.png", p3, width = 20, height = 6)
 


In [41]:
combined_psi_by_condition <- combined_psi_filtered %>% 
mutate(condition = gsub("MUT2_CH3-1_A\\d", "", Folder)) %>%
mutate(condition = str_extract(condition, "FUBP1|RBM10|RBM5|ZRSR2"))
combined_psi_by_condition_agg <- combined_psi_by_condition %>% 
group_by(ExonID, offset, skipped_exon_start, skipped_exon_end, downstream_exon_start, condition) %>% 
summarise(
    PSI1_average = mean(PSI1_average),
    PSI2_average = mean(PSI2_average),
    FDR = mean(FDR)) %>% 
mutate(PSI_diff = PSI1_average - PSI2_average) %>% 
mutate(PSI_ratio = PSI1_average / PSI2_average) %>% 
mutate(PSI_reverse_ratio = (1-PSI1_average)/(1-PSI2_average)) %>% 
mutate(log2_PSI_ratio = log2(PSI1_average / PSI2_average)) %>% 
mutate(log2_PSI_reverse_ratio = log2((1-PSI1_average)/(1-PSI2_average)))

p3 <- ggplot(combined_psi_by_condition_agg, aes(PSI_diff, -log10(FDR))) + 
geom_point() + 
facet_wrap(~condition, nrow = 1) +
theme_classic()

ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_volcano_plot_PSI_diff_by_condition.png", p3, width = 10, height = 3)

p4 <- ggplot(combined_psi_by_condition_agg, aes(log2_PSI_ratio, -log10(FDR))) + 
geom_point() + 
facet_wrap(~condition, nrow = 1) +
theme_classic()

ggsave("/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_volcano_plot_PSI_ratio_by_condition.png", p4, width = 10, height = 3)



`summarise()` has grouped output by 'ExonID', 'offset', 'skipped_exon_start',
'skipped_exon_end', 'downstream_exon_start'. You can override using the
`.groups` argument.


In [57]:
# Look at RBM10 and get the ones that are in both RBM10_C8 and RBM10_G4
rbm10_c8 <- combined_psi_filtered %>% 
filter(grepl("RBM10_C8", Folder)) %>% 
filter(FDR < 0.05) %>% 
pull(ExonID)

rbm10_g4 <- combined_psi_filtered %>% 
filter(grepl("RBM10_G4", Folder)) %>% 
filter(FDR < 0.05) %>% 
pull(ExonID)

rbm10_c8_g4 <- intersect(rbm10_c8, rbm10_g4)

# pull these sequences
rbm10_seqs <- combined_psi_filtered %>% 
filter(ExonID %in% rbm10_c8_g4) %>% 
arrange(desc(PSI_reverse_ratio))

rbm10_all <- combined_psi_filtered %>% 
filter(grepl("RBM10", Folder)) %>% 
arrange(desc(PSI_reverse_ratio))

# Look at sequences that appeared 4 times. 
rbm10_seqs_4 <- rbm10_all %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
filter(count >= 4) %>% 
pull(ExonID)

rbm10_high_confidence <- rbm10_all %>% 
filter(ExonID %in% rbm10_seqs_4) %>% 
arrange(ExonID)

# save these sequences
write_csv(rbm10_high_confidence, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_RBM10_high_confidence_seq.csv")



In [62]:
# Do the same for FUBP1
fubp1_all <- combined_psi_filtered %>% 
filter(grepl("FUBP1", Folder)) %>% 
arrange(desc(PSI_reverse_ratio))

# Look at sequences that appeared 4 times. 
fubp1_seqs_4 <- fubp1_all %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
filter(count >= 3) %>% 
pull(ExonID)

fubp1_high_confidence <- fubp1_all %>% 
filter(ExonID %in% fubp1_seqs_4) %>% 
arrange(ExonID)

# save these sequences
write_csv(fubp1_high_confidence, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_FUBP1_high_confidence_seq.csv")

In [63]:
# Do the same for RBM5
rbm5_all <- combined_psi_filtered %>% 
filter(grepl("RBM5", Folder)) %>% 
arrange(desc(PSI_reverse_ratio))

# Look at sequences that appeared 4 times. 
rbm5_seqs_4 <- rbm5_all %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
filter(count >= 3) %>% 
pull(ExonID)

rbm5_high_confidence <- rbm5_all %>% 
filter(ExonID %in% rbm5_seqs_4) %>% 
arrange(ExonID)

# save these sequences
write_csv(rbm5_high_confidence, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_RBM5_high_confidence_seq.csv")


In [64]:
# Do the same for ZRSR2
zrsr2_all <- combined_psi_filtered %>% 
filter(grepl("ZRSR2", Folder)) %>% 
arrange(desc(PSI_reverse_ratio))

# Look at sequences that appeared 4 times. 
zrsr2_seqs_4 <- zrsr2_all %>% 
group_by(ExonID) %>% 
summarise(count = n()) %>% 
filter(count >= 3) %>% 
pull(ExonID)

zrsr2_high_confidence <- zrsr2_all %>% 
filter(ExonID %in% zrsr2_seqs_4) %>% 
arrange(ExonID)

# save these sequences
write_csv(zrsr2_high_confidence, "/mnt/dawnccle2/processed_data/reprocess_250221/pairadise_indiv_PSI/seq_output/MUT2_ZRSR2_high_confidence_seq.csv")